# 02 — Mesh and Constraint Mapping

Prepare geometry artifacts, generate the CalculiX mesh, and inspect the fixed/load node sets.

In [ ]:
from pathlib import Path
import json
import logging
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

sample_id = "00689964"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
STATE_B_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
SOURCE_STEP = STATE_B_DIR / "fea_revision.step"
GEOMETRY_SUMMARY_PATH = STATE_B_DIR / "geometry_summary.json"
MESH_SUMMARY_PATH = STATE_B_DIR / "mesh_summary.json"

print("[SETUP] complete")
print(f"  → sample_id : {sample_id}")
print(f"  → state B   : {STATE_B_DIR}")

In [ ]:
def _load_region_summary(payload: dict[str, object]) -> api.RegionSelectionSummary:
    region = payload["region_selection"]
    assert isinstance(region, dict)
    return api.RegionSelectionSummary(
        major_axis=str(region["major_axis"]),
        axis_index=int(region["axis_index"]),
        lower_threshold_mm=float(region["lower_threshold_mm"]),
        upper_threshold_mm=float(region["upper_threshold_mm"]),
        fixed_node_ids=[int(node_id) for node_id in region.get("fixed_node_ids", [])],
        load_node_ids=[int(node_id) for node_id in region.get("load_node_ids", [])],
    )

def _load_mesh_summary(path: Path) -> api.MeshSummary:
    payload = json.loads(path.read_text(encoding="utf-8"))
    return api.MeshSummary(
        inp_path=Path(payload["inp_path"]),
        preview_path=Path(payload["preview_path"]),
        summary_path=Path(payload["summary_path"]),
        node_count=int(payload["node_count"]),
        element_count=int(payload["element_count"]),
        element_type_counts={str(key): int(value) for key, value in payload.get("element_type_counts", {}).items()},
        region_selection=_load_region_summary(payload),
        geometry_step_path=Path(payload["geometry_step_path"]),
        mesh_size_mm=float(payload["mesh_size_mm"]),
    )

In [ ]:
print("[STAGE] geometry preparation")
config = api.build_baseline_config(
    run_dir=STATE_B_DIR,
    source_step_path=SOURCE_STEP,
    mesh_size_mm=5.0,
    load_magnitude_n=200.0,
)
if GEOMETRY_SUMMARY_PATH.exists():
    geometry_payload = json.loads(GEOMETRY_SUMMARY_PATH.read_text(encoding="utf-8"))
    geometry = api.GeometrySummary(
        source_mode=str(geometry_payload["source_mode"]),
        step_path=Path(geometry_payload["step_path"]),
        preview_path=Path(geometry_payload["preview_path"]),
        summary_path=Path(geometry_payload["summary_path"]),
        stl_path=Path(geometry_payload["stl_path"]) if geometry_payload.get("stl_path") else None,
        name=str(geometry_payload["name"]),
        face_count=int(geometry_payload["face_count"]),
        edge_count=int(geometry_payload["edge_count"]),
        solid_count=int(geometry_payload["solid_count"]),
        bbox_min_mm=tuple(float(value) for value in geometry_payload["bbox_min_mm"]),
        bbox_max_mm=tuple(float(value) for value in geometry_payload["bbox_max_mm"]),
        spans_mm=tuple(float(value) for value in geometry_payload["spans_mm"]),
        major_axis=str(geometry_payload["major_axis"]),
        volume_mm3=geometry_payload.get("volume_mm3"),
        surface_area_mm2=geometry_payload.get("surface_area_mm2"),
    )
else:
    geometry = api.prepare_geometry_artifacts(config, force=False)
print(f"  ✓ geometry step : {geometry.step_path}")
print(f"  ✓ geometry stl  : {geometry.stl_path}")
print(f"  ✓ summary path  : {geometry.summary_path}")
print(f"  ✓ preview path  : {geometry.preview_path}")
print(f"  ✓ major axis    : {geometry.major_axis}")
assert geometry.step_path.exists()
assert geometry.stl_path is not None and geometry.stl_path.exists()
assert geometry.summary_path.exists()

In [ ]:
print("[STAGE] mesh generation")
config = api.build_baseline_config(
    run_dir=STATE_B_DIR,
    source_step_path=SOURCE_STEP,
    mesh_size_mm=5.0,
    load_magnitude_n=200.0,
)
if MESH_SUMMARY_PATH.exists():
    mesh = _load_mesh_summary(MESH_SUMMARY_PATH)
else:
    geometry = api.prepare_geometry_artifacts(config, force=False)
    mesh = api.generate_calculix_mesh(config, geometry, force=False)
region = mesh.region_selection
print(f"  ✓ inp path     : {mesh.inp_path}")
print(f"  ✓ mesh summary : {mesh.summary_path}")
print(f"  ✓ preview path : {mesh.preview_path}")
print(f"  ✓ node count   : {mesh.node_count}")
print(f"  ✓ element count: {mesh.element_count}")
print(f"  ✓ fixed nodes  : {len(region.fixed_node_ids)}")
print(f"  ✓ load nodes   : {len(region.load_node_ids)}")
print(f"  ✓ element map  : {mesh.element_type_counts}")
assert mesh.node_count > 0
assert mesh.element_count > 0
assert len(region.fixed_node_ids) > 0
assert len(region.load_node_ids) > 0

In [ ]:
print("[STAGE] failure case")
try:
    config = api.build_baseline_config(
        run_dir=STATE_B_DIR,
        source_step_path=SOURCE_STEP,
        mesh_size_mm=5.0,
        load_magnitude_n=200.0,
    )
    geometry = api.prepare_geometry_artifacts(config, force=False)
    api.generate_calculix_mesh(config, geometry, force=False)
except Exception as exc:
    print(f"  ✓ error type : {type(exc).__name__}")
    print(f"  ✓ message    : {exc}")

## Summary

- Geometry and mesh artifacts are visible through the public replication interface.
- Fixed and load node-set counts are shown from the mesh summary.
- The failure case makes the overwrite guard visible.